<a href="https://colab.research.google.com/github/annanya-sharma1002/AI_for_Health/blob/main/breathing_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt
import pickle
from datetime import timedelta

# --- FILTER FUNCTION ---
def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    # 'btype=band' is used for bandpass
    b, a = butter(order, [low, high], btype='band')
    # Use fillna to prevent filter crash if data has gaps
    clean_data = np.nan_to_num(data)
    return filtfilt(b, a, clean_data)

def create_dataset(data_dir):
    all_windows = []
    fs_respiratory = 32
    window_sec = 30
    overlap_sec = 15

    # Loop through participant folders in 'Data'
    participants = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)) and not d.startswith('.')]

    for participant in participants:
        p_path = os.path.join(data_dir, participant)
        print(f"🔄 Processing {participant}...")

        # Load CSVs
        nasal = pd.read_csv(os.path.join(p_path, "Flow.csv"), parse_dates=['timestamp'])
        thoracic = pd.read_csv(os.path.join(p_path, "Thorac.csv"), parse_dates=['timestamp'])
        spo2 = pd.read_csv(os.path.join(p_path, "SPO2.csv"), parse_dates=['timestamp'])
        events = pd.read_csv(os.path.join(p_path, "Flow_Events.csv"))
        events['start_time'] = pd.to_datetime(events['start_time'])
        events['end_time'] = pd.to_datetime(events['start_time'].dt.date.astype(str) + ' ' + events['end_time'])

        # Apply Filtering (0.1Hz - 0.5Hz)
        print("   Filtering signals...")
        n_filt = butter_bandpass_filter(nasal['value'].values, 0.1, 0.5, fs_respiratory)
        t_filt = butter_bandpass_filter(thoracic['value'].values, 0.1, 0.5, fs_respiratory)

        # Create indexed Series for fast windowing
        nasal['filtered'] = n_filt
        thoracic['filtered'] = t_filt

        start_time = nasal['timestamp'].min()
        end_time = nasal['timestamp'].max()

        print("   Windowing and Labeling...")
        current_start = start_time
        while current_start + timedelta(seconds=window_sec) <= end_time:
            current_end = current_start + timedelta(seconds=window_sec)

            # Extract signal segments
            n_win = nasal[(nasal['timestamp'] >= current_start) & (nasal['timestamp'] < current_end)]['filtered'].values
            t_win = thoracic[(thoracic['timestamp'] >= current_start) & (thoracic['timestamp'] < current_end)]['filtered'].values
            s_win = spo2[(spo2['timestamp'] >= current_start) & (spo2['timestamp'] < current_end)]['value'].values

            # Labeling Logic (50% overlap rule)
            label = "Normal"
            # Find events that happen during this 30s window
            mask = (events['start_time'] < current_end) & (events['end_time'] > current_start)
            overlap_events = events[mask]

            for _, event in overlap_events.iterrows():
                o_start = max(current_start, event['start_time'])
                o_end = min(current_end, event['end_time'])
                duration = (o_end - o_start).total_seconds()

                if duration > 15: # More than 50% of 30s
                    label = event['events']
                    break

            all_windows.append({
                'participant': participant,
                'label': label,
                'nasal': n_win,
                'thoracic': t_win,
                'spo2': s_win
            })

            current_start += timedelta(seconds=overlap_sec)

    # Save to Dataset folder
    os.makedirs("Dataset", exist_ok=True)
    output_file = "Dataset/processed_dataset.pkl"
    with open(output_file, "wb") as f:
        pickle.dump(all_windows, f)

    print(f"✅ DONE! Saved {len(all_windows)} windows to {output_file}")

# Run it
create_dataset("/content/drive/MyDrive/Data")

🔄 Processing AP01...


/tmp/ipykernel_415/1692917073.py:33: UserWarning: Parsing dates in %d.%m.%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  nasal = pd.read_csv(os.path.join(p_path, "Flow.csv"), parse_dates=['timestamp'])
/tmp/ipykernel_415/1692917073.py:34: UserWarning: Parsing dates in %d.%m.%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  thoracic = pd.read_csv(os.path.join(p_path, "Thorac.csv"), parse_dates=['timestamp'])
/tmp/ipykernel_415/1692917073.py:35: UserWarning: Parsing dates in %d.%m.%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  spo2 = pd.read_csv(os.path.join(p_path, "SPO2.csv"), parse_dates=['timestamp'])
/tmp/ipykernel_415/1692917073.py:37: UserWarning: Parsing dates in %d.%m.%Y %H:%M:%S format when dayfirst=False (the default) 

   Filtering signals...
   Windowing and Labeling...
✅ DONE! Saved 1822 windows to Dataset/processed_dataset.pkl
